<a href="https://colab.research.google.com/github/Pra1m3/nyc-housing-repo-milestone1/blob/main/notebooks/M1_Housing_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Replaced defaults with the dataset IDs found in each dataset's URL(H2 and T4)
housing_id = "hg8x-zxpr"
transport_id = "xeqp-qz8h"

base_url = "https://data.cityofnewyork.us/api/views/{}/rows.csv?accessType=download"

# Loading chosen datasets
df_housing = pd.read_csv(base_url.format(housing_id), low_memory = False )
df_transport = pd.read_csv(base_url.format(transport_id), low_memory = False )

# Preview
df_transport.head()
df_housing.head()

,Project ID,Project Name,Project Start Date,Project Completion Date,Building ID,Number,Street,Borough,Postcode,BBL,...,2-BR Units,3-BR Units,4-BR Units,5-BR Units,6-BR+ Units,Unknown-BR Units,Counted Rental Units,Counted Homeownership Units,All Counted Units,Total Units
0,68872,177-06 WEXFORD TERRACE,06/30/2026,NaN,991637.0,177-06,WEXFORD TERRACE,Queens,11432.0,4.098350e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,9.0,NaN,9.0,43
1,72436,ST. NICK'S. 819 GRAND STREET,06/30/2026,NaN,1021347.0,819,GRAND STREET,Brooklyn,11211.0,3.029220e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,187.0,NaN,187.0,187
2,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,21199.0,127,EDGECOMBE AVENUE,Manhattan,10030.0,1.020480e+09,...,20.0,NaN,NaN,NaN,NaN,NaN,25.0,NaN,25.0,25
3,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,28145.0,602,ST NICHOLAS AVENUE,Manhattan,10030.0,1.020480e+09,...,5.0,NaN,NaN,NaN,NaN,NaN,16.0,NaN,16.0,16
4,72449,BREMOND KING DAVIS.YR15.FY26,06/30/2026,NaN,40555.0,129,WEST 133 STREET,Manhattan,10030.0,1.019180e+09,...,6.0,4.0,NaN,NaN,NaN,NaN,15.0,NaN,15.0,15


In [2]:
#Remove exact duplicate rows only, added .copy() for given error but can be removed
df_housing_clean = df_housing.drop_duplicates().copy()
df_transport_clean = df_transport.drop_duplicates().copy()

#Convert text dates to real dates
for c in ["Project Start Date", "Project Completion Date"]:
    df_housing_clean[c] = pd.to_datetime(df_housing_clean[c], format="%m/%d/%Y", errors="coerce")

print(df_housing_clean["Project Start Date"].dtype)
print(df_housing_clean["Project Start Date"].isna().sum(), "unparsed start dates")

date_col = "Crash Date"
df_transport_clean[date_col] = pd.to_datetime(df_transport_clean[date_col], errors="coerce")

#Converting the 5-digit ZIP to text for standardization, dropping decimals and extra characters
df_housing_clean["Postcode"] = df_housing_clean["Postcode"].astype("string").str.extract(r"(\d{5})")[0]

#Flag unusually large buildings using 1.5xIQR rule
col = "Total Units"
q1, q3 = df_housing_clean[col].quantile([0.25, 0.75])
iqr = q3 - q1
df_housing_clean["units_outlier"] = ~df_housing_clean[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

#Show row counts before and after cleaning
print(len(df_housing), "->", len(df_housing_clean))
print(len(df_transport), "->", len(df_transport_clean))
print(df_housing_clean["units_outlier"].sum(), "flagged outlier buildings")

datetime64[ns]
0 unparsed start dates
9551 -> 9551
13754 -> 13751
1171 flagged outlier buildings


In [3]:
#Removing un-groupable rows(no ZIP or start date)
df_housing_clean = df_housing_clean.dropna(subset=["Postcode", "Project Start Date"])
df_housing_clean["year"] = df_housing_clean["Project Start Date"].dt.year

housing_zip_year = (
    df_housing_clean.groupby(["Postcode", "year"])
    .agg(buildings=("Building ID", "count"),
         total_units=("Total Units", "sum"),
         counted_units=("All Counted Units", "sum"))
    .reset_index()
    .rename(columns={"Postcode": "zip"})
)

#Print rows and columns
print(housing_zip_year.shape)

(1155, 5)


In [4]:
#Check for missed ZIPs/ any other units that may have been looked over by code
print(housing_zip_year["year"].min(), housing_zip_year["year"].max())
print(housing_zip_year["zip"].nunique(), "ZIPs")
print(housing_zip_year.isna().sum())
housing_zip_year.to_csv("housing_zip_year.csv", index=False)

2014 2026
158 ZIPs
zip              0
year             0
buildings        0
total_units      0
counted_units    0
dtype: int64


In [5]:
#saving to CSV
housing_zip_year.to_csv("housing_zip_year.csv", index=False)